# 程式組加碼：100 則員工意見調查批次分析（參考答案）
東吳資科系 × SCU NLP Lab｜AI 職場溝通革命工作坊 2026/11/07

這份筆記本是「請 Claude 寫程式 → 自己在 Colab 執行」的參考答案。**不需要 API 金鑰**，全部在 Colab 免費版就能跑。

資料是虛構公司「禾川精密」的員工意見調查：讀 `employee_survey.csv` → 依關鍵字分類 → 統計各類別與廠區 → 畫圖 → 匯出 Excel。

> 💡 課堂上你應該先自己請 Claude 寫一版，再回來對照這份。

## 1. 讀取資料
先試著從課程網站下載；下載不到就請你手動上傳 `employee_survey.csv`。

In [ ]:
import pandas as pd

DATA_URL = "https://jlwustudio.github.io/chat2agent_workshop_2026/data/employee_survey.csv"

try:
    df = pd.read_csv(DATA_URL)
except Exception:
    try:
        df = pd.read_csv("employee_survey.csv")
    except FileNotFoundError:
        from google.colab import files  # 只有在 Colab 才有
        files.upload()
        df = pd.read_csv("employee_survey.csv")

print(len(df), "則意見")
df.head()

## 2. 依關鍵字分類
規則式分類的好處是**每一筆都說得出為什麼**，可以和 AI 分類的結果對照。

In [ ]:
RULES = {
    "排班與加班": ["加班", "排班", "夜班", "輪調"],
    "薪酬福利": ["調薪", "薪", "獎金", "餐廳", "宿舍"],
    "主管溝通": ["主管", "決策", "開會", "反映"],
    "設備與安全": ["治具", "設備", "走道", "危險", "防護", "冷氣"],
    "訓練與發展": ["訓練", "進修", "升遷", "AI"],
}

def classify(text):
    for cat, words in RULES.items():
        if any(w in text for w in words):
            return cat
    return "其他"

df["category"] = df["text"].apply(classify)
df["sentiment"] = df["score"].apply(lambda s: "不滿意" if s <= 2 else ("滿意" if s >= 4 else "普通"))
df[["response_id", "site", "role", "score", "text", "category", "sentiment"]].head(10)

## 3. 統計

In [ ]:
summary = (df.groupby("category")
             .agg(則數=("response_id", "count"), 平均分數=("score", "mean"),
                  不滿意數=("sentiment", lambda s: (s == "不滿意").sum()))
             .sort_values("則數", ascending=False)
             .round(2))
summary

In [ ]:
by_site = pd.crosstab(df["site"], df["category"])
by_site["不滿意合計"] = df[df["sentiment"] == "不滿意"].groupby("site").size()
by_site = by_site.fillna(0).astype(int).sort_values("不滿意合計", ascending=False)
by_site

## 4. 畫圖

In [ ]:
import matplotlib.pyplot as plt
import matplotlib

# Colab 預設沒有中文字型：下載思源黑體（Noto Sans TC）
try:
    import urllib.request, matplotlib.font_manager as fm, os
    font_path = "NotoSansTC.ttf"
    if not os.path.exists(font_path):
        urllib.request.urlretrieve(
            "https://github.com/google/fonts/raw/main/ofl/notosanstc/NotoSansTC%5Bwght%5D.ttf", font_path)
    fm.fontManager.addfont(font_path)
    matplotlib.rcParams["font.family"] = fm.FontProperties(fname=font_path).get_name()
except Exception as e:
    print("中文字型下載失敗，圖上的中文可能變方塊：", e)

ax = summary["則數"].plot(kind="barh", color="#DA7756", figsize=(7, 4))
ax.invert_yaxis()
ax.set_title("10 月員工意見：問題類別")
ax.set_xlabel("則數")
ax.set_ylabel("")
plt.tight_layout()
plt.savefig("survey_categories.png", dpi=150)
plt.show()

## 5. 匯出 Excel
執行後在左側「檔案」面板下載 `survey_analysis.xlsx`。

In [ ]:
with pd.ExcelWriter("survey_analysis.xlsx") as w:
    summary.to_excel(w, sheet_name="類別統計")
    by_site.to_excel(w, sheet_name="廠區統計")
    df.to_excel(w, sheet_name="原始資料與分類", index=False)
print("已輸出 survey_analysis.xlsx")

## 6. 🔍 追問檢查點：規則分類可靠嗎？
1. 執行最下面的格子，看哪些意見被分到「其他」。如果是空的，自己加一則「停車位不夠」再跑一次，看規則漏了什麼。
2. 「宿舍冷氣壞了」該算薪酬福利還是設備與安全？規則先比對到哪一類，就會被分到哪一類——這合理嗎？
3. 一則意見同時講「加班」和「調薪」，只能分一類合理嗎？分數和文字不一致時該信哪個？

這些問題——**怎麼評估分類對不對、怎麼處理一則多類、怎麼讓 AI 和規則互相檢查**——就是碩士班 NLP 與機器學習課在做的事。

In [ ]:
df[df["category"] == "其他"]